In [1]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("RDD Transformation")
    .master("local[*]")
    .getOrCreate()
)
sc=spark.sparkContext
print(sc.uiWebUrl)

sc

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/18 20:37:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://macbookair.lan:4040


<SparkContext master=local[*] appName=RDD Transformation>

# Narrow Transformation 

In [ ]:
# map() - This is an RDD transformation that applies a function to each element of an RDD and returns exactly one output element for each input element.

# map() -> 1 input -> 1 output 

#new_rdd=rdd.map(function)

numbers=[10,20,30,40,50]
rdd=sc.parallelize(numbers)
result=rdd.map(lambda x:x*2)
result.collect()


In [ ]:
def multiple_by_10(x):
    return x*10

In [ ]:
result1=rdd.map(multiple_by_10)
result1.collect()

In [ ]:
# Important 

rdd=sc.parallelize([
    "Vijay",
    "Vamshi",
    "Darshan",
    "Ashad"
])

In [ ]:
result=rdd.map(lambda name:(name,len(name)))

In [ ]:
result.collect()

In [ ]:
row_data=[
    "T001,C101,5000,India",
    "T002,C102,6000,Usa",
    "T003,C103,59000,uk",
    "T004,C104,5400,Usa",
    "T005,C105,50400,India"

]

In [ ]:
rdd_row_data=sc.parallelize(row_data)
result_row_data=rdd_row_data.map(lambda x:x.split(","))
result_row_data.collect()

In [ ]:
def parse_transformation(record):
    txn_id,customer_id,amount,country=record.split(",")
    return(
        txn_id,
        customer_id,
        float(amount),
        country.upper()
    )

In [ ]:
transection_rdd=rdd_row_data.map(parse_transformation)


In [ ]:
transection_rdd.collect()

### Map() Usecases 

- Parsing records 
- Type Conversion 
- Extacting Values 
- Adding calculated fields 
- Converting one struture into another 
- Creating key-value RDD
- Applying Bsuiness Logic 



In [ ]:
transections=[
    ('T001', 'C101', 5000.0, 'INDIA'),
    ('T002', 'C102', 6000.0, 'USA'),
    ('T003', 'C103', 59000.0, 'UK'),
    ('T004', 'C104', 5400.0, 'USA'),
    ('T005', 'C105', 50400.0, 'INDIA')]
rdd=sc.parallelize(transections)

In [ ]:
customer_amount=rdd.map(lambda x:(x[1],x[2]))
customer_amount.collect()

In [ ]:
rdd=sc.parallelize([1,2,3])
result=rdd.map(lambda x:[x,x*10])
result.collect()

In [ ]:
# Map() and Task Execution 

rdd=sc.parallelize(range(1_000_000),4)
result=rdd.map(lambda x:x*2)
result.collect()

In [ ]:
# Multiple Narrow Transformation 

rdd=sc.parallelize(range(100),4)
rdd2=rdd.map(lambda x:x*2)
rdd3=rdd2.map(lambda x:x*10)
rdd4=rdd3.filter(lambda x:x>50)
rdd4.collect()

## Questions map() Transformation 

Q1 — Employee Salary Hike

employees = [
    ("E101", "An", "Data Engineer", 80000),
    ("E102", "Rahul", "Developer", 60000),
    ("E103", "Priya", "Data Engineer", 90000),
    ("E104", "Amit", "Tester", 50000)
]

rdd = sc.parallelize(employees)


- Using only map(), apply these hikes:

Data Engineer → 15%
Developer     → 10%
Tester        → 5%

(employee_id, name, role, old_salary, new_salary)


Q2 — Transaction Risk Classification

transactions = [
    ("T001", "C101", 5000, "India"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "India"),
    ("T004", "C104", 25000, "UK"),
    ("T005", "C105", 120000, "USA")
]

rdd = sc.parallelize(transactions)

Rules: 

amount < 10,000              → LOW
10,000 <= amount < 50,000    → MEDIUM
amount >= 50,000             → HIGH

Expected Structure:

(transaction_id, customer_id, amount, country, risk)

Q3 — Parse Raw CSV Records 

raw_data = [
    "E101,Anuj,Data Engineer,85000,India",
    "E102,Rahul,Developer,65000,USA",
    "E103,Priya,Manager,120000,India"
]

rdd = sc.parallelize(raw_data)


- Using map(), convert each string into a tuple and convert salary to int.
- Also uppercase the country.


Q4 — E-commerce Order Calculation

orders = [
    ("O101", "Laptop", 2, 50000),
    ("O102", "Mouse", 5, 1000),
    ("O103", "Keyboard", 3, 2000),
    ("O104", "Monitor", 2, 15000)
]

rdd = sc.parallelize(orders)

Fields:

(order_id, product, quantity, unit_price)

Calculate:

total_amount = quantity × unit_price
GST          = total_amount × 18%
final_amount = total_amount + GST

Expected Structure:

(order_id, product, total_amount, gst, final_amount)

-- Example :

("O101", "Laptop", 100000, 18000, 118000)
("O102", "Mouse",    5000,   900,   5900)


Q5 — Create a Pair RDD for Future Aggregation

sales = [
    ("S001", "C101", "Laptop", 50000),
    ("S002", "C102", "Mobile", 30000),
    ("S003", "C101", "Keyboard", 5000),
    ("S004", "C103", "Monitor", 20000),
    ("S005", "C102", "Mouse", 2000)
]

rdd = sc.parallelize(sales)

Transform this into a Pair RDD where:

Key   = Customer ID
Value = (Product, Amount)


Expected:

("C101", ("Laptop", 50000))
("C102", ("Mobile", 30000))
("C101", ("Keyboard", 5000))
("C103", ("Monitor", 20000))
("C102", ("Mouse", 2000))

Q6 — Data Quality Flagging

customers = [
    ("C101", "Anuj", 32, "anuj@gmail.com"),
    ("C102", "", 28, "rahul@gmail.com"),
    ("C103", "Priya", -5, "priya@gmail.com"),
    ("C104", "Amit", 45, ""),
    ("C105", "Neha", 25, "neha@gmail.com")
]

rdd = sc.parallelize(customers)


Rules:

name is empty  → INVALID_NAME

age <= 0       → INVALID_AGE

email is empty → INVALID_EMAIL

otherwise      → VALID


Expected Struccure:

(customer_id, name, age, email, status)

Expected:

("C101", "Anuj",  32, "anuj@gmail.com",  "VALID")
("C102", "",      28, "rahul@gmail.com", "INVALID_NAME")
("C103", "Priya", -5, "priya@gmail.com", "INVALID_AGE")
("C104", "Amit",  45, "",                 "INVALID_EMAIL")
("C105", "Neha",  25, "neha@gmail.com",  "VALID")


Q7 — Multiple Business Rules

transactions = [
    ("T001", "C101", 5000, "India", "UPI"),
    ("T002", "C102", 60000, "India", "CARD"),
    ("T003", "C103", 120000, "USA", "CARD"),
    ("T004", "C104", 8000, "UK", "CASH"),
    ("T005", "C105", 90000, "India", "UPI")
]

rdd = sc.parallelize(transactions)

Create :

(transaction_id,
 customer_id,
 amount,
 country,
 payment_method,
 risk_score,
 risk_category)

 Risk Score:

 amount >= 100000        → +3
amount >= 50000         → +2
otherwise               → +1

country != "India"      → +2

payment_method == CARD  → +1

Risk category:

score >= 5 → HIGH
score >= 3 → MEDIUM
otherwise  → LOW


Example:

T003
Amount 120000 → +3
USA           → +2
CARD          → +1
                 --
Score            6

Risk = HIGH


Expected T003: ("T003", "C103", 120000, "USA", "CARD", 6, "HIGH")






In [ ]:
# ## Questions map() Transformation 

# Q1 — Employee Salary Hike

# employees = [
#     ("E101", "An", "Data Engineer", 80000),
#     ("E102", "Rahul", "Developer", 60000),
#     ("E103", "Priya", "Data Engineer", 90000),
#     ("E104", "Amit", "Tester", 50000)
# ]

# rdd = sc.parallelize(employees)


# - Using only map(), apply these hikes:

# Data Engineer → 15%
# Developer     → 10%
# Tester        → 5%

# (employee_id, name, role, old_salary, new_salary)


# Q2 — Transaction Risk Classification

# transactions = [
#     ("T001", "C101", 5000, "India"),
#     ("T002", "C102", 18000, "USA"),
#     ("T003", "C103", 75000, "India"),
#     ("T004", "C104", 25000, "UK"),
#     ("T005", "C105", 120000, "USA")
# ]

# rdd = sc.parallelize(transactions)

# Rules: 

# amount < 10,000              → LOW
# 10,000 <= amount < 50,000    → MEDIUM
# amount >= 50,000             → HIGH

# Expected Structure:

# (transaction_id, customer_id, amount, country, risk)

# Q3 — Parse Raw CSV Records 

# raw_data = [
#     "E101,Anuj,Data Engineer,85000,India",
#     "E102,Rahul,Developer,65000,USA",
#     "E103,Priya,Manager,120000,India"
# ]

# rdd = sc.parallelize(raw_data)


# - Using map(), convert each string into a tuple and convert salary to int.
# - Also uppercase the country.


# Q4 — E-commerce Order Calculation

# orders = [
#     ("O101", "Laptop", 2, 50000),
#     ("O102", "Mouse", 5, 1000),
#     ("O103", "Keyboard", 3, 2000),
#     ("O104", "Monitor", 2, 15000)
# ]

# rdd = sc.parallelize(orders)

# Fields:

# (order_id, product, quantity, unit_price)

# Calculate:

# total_amount = quantity × unit_price
# GST          = total_amount × 18%
# final_amount = total_amount + GST

# Expected Structure:

# (order_id, product, total_amount, gst, final_amount)

# -- Example :

# ("O101", "Laptop", 100000, 18000, 118000)
# ("O102", "Mouse",    5000,   900,   5900)


# Q5 — Create a Pair RDD for Future Aggregation

# sales = [
#     ("S001", "C101", "Laptop", 50000),
#     ("S002", "C102", "Mobile", 30000),
#     ("S003", "C101", "Keyboard", 5000),
#     ("S004", "C103", "Monitor", 20000),
#     ("S005", "C102", "Mouse", 2000)
# ]

# rdd = sc.parallelize(sales)

# Transform this into a Pair RDD where:

# Key   = Customer ID
# Value = (Product, Amount)


# Expected:

# ("C101", ("Laptop", 50000))
# ("C102", ("Mobile", 30000))
# ("C101", ("Keyboard", 5000))
# ("C103", ("Monitor", 20000))
# ("C102", ("Mouse", 2000))

# Q6 — Data Quality Flagging

# customers = [
#     ("C101", "Anuj", 32, "anuj@gmail.com"),
#     ("C102", "", 28, "rahul@gmail.com"),
#     ("C103", "Priya", -5, "priya@gmail.com"),
#     ("C104", "Amit", 45, ""),
#     ("C105", "Neha", 25, "neha@gmail.com")
# ]

# rdd = sc.parallelize(customers)


# Rules:

# name is empty  → INVALID_NAME

# age <= 0       → INVALID_AGE

# email is empty → INVALID_EMAIL

# otherwise      → VALID


# Expected Struccure:

# (customer_id, name, age, email, status)

# Expected:

# ("C101", "Anuj",  32, "anuj@gmail.com",  "VALID")
# ("C102", "",      28, "rahul@gmail.com", "INVALID_NAME")
# ("C103", "Priya", -5, "priya@gmail.com", "INVALID_AGE")
# ("C104", "Amit",  45, "",                 "INVALID_EMAIL")
# ("C105", "Neha",  25, "neha@gmail.com",  "VALID")


# Q7 — Multiple Business Rules

# transactions = [
#     ("T001", "C101", 5000, "India", "UPI"),
#     ("T002", "C102", 60000, "India", "CARD"),
#     ("T003", "C103", 120000, "USA", "CARD"),
#     ("T004", "C104", 8000, "UK", "CASH"),
#     ("T005", "C105", 90000, "India", "UPI")
# ]

# rdd = sc.parallelize(transactions)

# Create :

# (transaction_id,
#  customer_id,
#  amount,
#  country,
#  payment_method,
#  risk_score,
#  risk_category)

#  Risk Score:

#  amount >= 100000        → +3
# amount >= 50000         → +2
# otherwise               → +1

# country != "India"      → +2

# payment_method == CARD  → +1

# Risk category:

# score >= 5 → HIGH
# score >= 3 → MEDIUM
# otherwise  → LOW


# Example:

# T003
# Amount 120000 → +3
# USA           → +2
# CARD          → +1
#                  --
# Score            6

# Risk = HIGH


# Expected T003: ("T003", "C103", 120000, "USA", "CARD", 6, "HIGH")




# # 

In [ ]:
# flatmap() : Applies a function to every element in RDD but one input element can produce zero,one or multiple output element.

# map() - I input -> 1 Output 

# flatmap() - 1 Input -> 0,1, or many outputs 

# flatmap() -> MAP() -> Transform each element   Flat-> Flatten the results 



In [ ]:
rdd=sc.parallelize([
    "Apache Spark",
    "Data Engineering",
    "Big Data"
])

In [ ]:
result=rdd.map(lambda x:x.split(" "))
result.collect()

In [ ]:
result=rdd.flatMap(lambda x:x.split(" "))
result.collect()

In [ ]:
rdd=sc.parallelize([
    "Spark",
    "",
    "Python"
])

In [ ]:
result=rdd.flatMap(lambda x:[x] if x != "" else [])

In [ ]:
result.collect()

In [ ]:
# Does flatMap() changes the Number of partition - No 

rdd=sc.parallelize([
    "Apache Spark",
    "Data Engineering",
    "Big Data"
],3)

rdd.getNumPartitions()




In [ ]:
result=rdd.flatMap(lambda x:x.split(" "))
result.getNumPartitions()

In [ ]:
result.collect()

In [ ]:
logs=[
    "ERROR database connection Failed",
    "INFO application started",
    "ERROR payment service timeout"
]

rdd=sc.parallelize(logs)

In [ ]:
words=rdd.flatMap(lambda line:line.split())

In [ ]:
words.collect()

In [ ]:
orders=[
    ("0101",["Laptop","Mouse"]),
    ("0102",["Keyboard"]),
    ("0103",["Monitor","Mouse","Keyboard"])
]

rdd=sc.parallelize(orders)

In [ ]:
result=rdd.flatMap(
    lambda x:[(x[0],product) for product in x[1]]
)

result.collect()

In [ ]:
customer=[
    ("C101",[5000,3000,7000]),
    ("C102",[10000,20000]),
    ("C103",[]),

]
rdd=sc.parallelize(customer)

In [ ]:
result=rdd.flatMap(
    lambda x:[(x[0],amount) for amount in x[1]]
)
result.collect()

In [ ]:
x=("C101",[])
output=[]
for amount in x[1]:
    output.append((x[0],amount))

In [ ]:
output

In [ ]:
# filter() : Returns only those RDD elements that staisfy a condition 

# new_rdd=rdd.filter(function)

# new_rdd=rdd.filter(lambda x: condition)

rdd=sc.parallelize([10,15,20,25,30])
result=rdd.filter(lambda x:x>20)
result.collect()



In [ ]:
rdd=sc.parallelize([1,2,3,4,5,6,7,7,8,8,8,78,6,6,56,5,4,4,3,3,3])

In [ ]:
result=rdd.filter(lambda x:x%2==0)
result.collect()

In [ ]:
logs=sc.parallelize([
    "INFO Application started",
    "ERROR Database connection failed",
    "WARN memory useage high",
    "ERROR payment service timeout",
    "INFO Application completed"
])



In [ ]:
errors=logs.filter(
    lambda line : "ERROR" in line
)

errors.collect()

# ── WIDE TRANSFORMATIONS
SET / DATASET
distinct()
intersection()
subtract()

In [ ]:
# distinct() : Removes duplicate elemnts from an RDD

# distinct(numPartitions) : The desired number of partition for the result \

-   rdd.distinct(numPartitions=10)

In [ ]:
rdd=sc.parallelize([
    10,20,10,30,20,40
])
result=rdd.distinct()
result.collect()

In [ ]:
events=sc.parallelize([
    ("U101","Product_1"),
    ("U102","Product_2"),
    ("U101","Product_1"),
    ("U103","Product_3"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U101","Product_1"),
])

# Find unique (user,product) interatction 

unique_events=events.distinct()
unique_events.collect()


1 TB -> 500 Partitions 

rdd.distinct()

Spark may need significant:

- Network I/O 
- Disk I/O
- Serialization 
- Shuffle Read/write 
- memory 

### Intersection - Finds the elements thta are present in both RDDs

RDD_1 and RDD_2 --> Common element 

rdd1.intersection(rdd2)





subtract() : - Returns elements that are present in the RDD but not in the second RDD

rdd.subtract(rdd2)

A-B - In A but Not B 

### Aggregation : 

- groupby()
- reduceBykey()
- groupBykey()
- aggregateByKey()
- combineByKey()
- foldByKey() 




### groupby()

- Create a Key using a function -> Bring records having the same key togather

- GroupBy() groups RDD elements based on a key that we calculate using a function 

- groupBy(function) -> Function return Value -> Become the group Key 





In [ ]:
rdd=sc.parallelize([
    10,15,20,25,30,35
])

# Even No togather and Odd no togather 



In [4]:
result=rdd.groupBy(
    lambda x:"EVEN" if x % 2==0 else "ODD"
)
#result.collect()
for key,values in result.collect():
    print(key,list(values))

ODD [15, 25, 35]
EVEN [10, 20, 30]


In [7]:
# Example 1- groupBy()

transactions = sc.parallelize([
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "INDIA"),
    ("T004", "C104", 120000, "UK"),
    ("T005", "C105", 35000, "INDIA"),
    ("T006", "C106", 95000, "USA"),
    ("T007", "C107", 8000, "UK")
], 3)

# Requirment 

# - Group Transection 
    # - LOW -> amount < 10000
    # - MEDIUM -> 10000 <= amount < 50000
    # - HIGH 5000 <= amount < 100000
    # - CRITICAL - amount >=100000

In [5]:
def get_amount_category(record):
    amount=record[2]
    if amount >=100000:
        return "CRITICAL"
    elif amount >=50000:
        return "HIGH"
    elif amount >=10000:
        return "MEDIUM"
    else:
        return "LOW"


In [8]:
result=transactions.groupBy(get_amount_category)

In [10]:
for category,records in result.collect():
    print("\n",category)
    for record in records:
        print(record)


 HIGH
('T003', 'C103', 75000, 'INDIA')
('T006', 'C106', 95000, 'USA')

 LOW
('T001', 'C101', 5000, 'INDIA')
('T007', 'C107', 8000, 'UK')

 MEDIUM
('T002', 'C102', 18000, 'USA')
('T005', 'C105', 35000, 'INDIA')

 CRITICAL
('T004', 'C104', 120000, 'UK')


In [11]:
# Example :2 
logs = sc.parallelize([
    ("2026-08-18 10:01", "payment", "ERROR", "Database connection timeout"),
    ("2026-08-18 10:02", "order", "ERROR", "API connection timeout"),
    ("2026-08-18 10:03", "payment", "ERROR", "Invalid authentication token"),
    ("2026-08-18 10:04", "customer", "ERROR", "Database connection refused"),
    ("2026-08-18 10:05", "order", "ERROR", "Out of memory"),
    ("2026-08-18 10:06", "payment", "ERROR", "Authentication failed")
], 3)

# Requirments :

# message contains "Database"
# -> database_Error 

# message contains "timeout"
# -> TIMEOUT_ERROR

# Message contains "authentication"
# - AUTH_ERROR

# Message Contains "memory"
# -> "MEMORY_ERROR"

# otherwise 
# -> Other_ERROR 



In [12]:
def classify_error(record):
    message=record[3].lower()

    if "database" in message:
        return "DATABASE_ERROR"
    elif "timeout" in message:
        return "TIMEOUT_ERROR"
    elif "authentication" in message:
        return "AUTH_ERROR"
    elif "memory" in message:
        return "MEMORY_ERROR"
    else:
        return "OTHER_ERROR"
    

In [13]:
result=logs.groupBy(classify_error)


In [14]:
for category,records in result.collect():
    print("\n",category)
    for record in records:
        print(record)


 AUTH_ERROR
('2026-08-18 10:03', 'payment', 'ERROR', 'Invalid authentication token')
('2026-08-18 10:06', 'payment', 'ERROR', 'Authentication failed')

 DATABASE_ERROR
('2026-08-18 10:01', 'payment', 'ERROR', 'Database connection timeout')
('2026-08-18 10:04', 'customer', 'ERROR', 'Database connection refused')

 TIMEOUT_ERROR
('2026-08-18 10:02', 'order', 'ERROR', 'API connection timeout')

 MEMORY_ERROR
('2026-08-18 10:05', 'order', 'ERROR', 'Out of memory')


### Example : 3 



### - reduceBykey()
- Works on pair RDD (key,value)

- It combines all values belonging to the same key using a function 



In [15]:
#Example -1 ReduceByKey()

sales =sc.parallelize([
    ("INDIA",1000),
    ("USA",7000),
    ("INDIA",5000),
    ("US",3000),
    ("UK",9000),
    ("USA",6000),
    ("USA",5000),
    ("INDIA",3000),
    ("UK",2000),
])

# Requirment:

# Calculate total sales for each country

In [16]:
result=sales.reduceByKey(lambda x,y:x+y)

In [17]:
result.collect()

[('USA', 18000), ('US', 3000), ('UK', 11000), ('INDIA', 9000)]

In [18]:
# example 2: reduceByKey()

transactions = sc.parallelize([
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "INDIA"),
    ("T004", "C104", 120000, "UK"),
    ("T005", "C105", 35000, "INDIA"),
    ("T006", "C106", 95000, "USA"),
    ("T007", "C107", 8000, "UK")
], 3)

# Requirment: Calculate total transection amount by country

In [19]:
pair_rdd=transactions.map(
    lambda x:(x[3],x[2])
)

In [21]:
def add_values(x,y):
    return x+y

In [22]:
result=pair_rdd.reduceByKey(add_values)

result.collect()

[('UK', 128000), ('INDIA', 115000), ('USA', 113000)]

# ============================================================
# PYSPARK RDD PRACTICE
#
## Topics you may need:
## distinct()
## union()
## intersection()
## subtract()
## filter()
## groupBy()
## reduceByKey()
## map()
## flatMap()
#
##Goal:
## Read the requirement and decide yourself which
## transformations are needed.
## ============================================================


## ============================================================
# QUESTION 1
# Consolidate Failed Customers Across Two Days
# Level: MEDIUM
# ============================================================

day1 = sc.parallelize([
    ("T001", "C101", 1200, "SUCCESS"),
    ("T002", "C102", 2500, "FAILED"),
    ("T003", "C103", 1800, "FAILED"),
    ("T004", "C104", 3200, "SUCCESS"),
    ("T005", "C105", 900,  "SUCCESS")
])

day2 = sc.parallelize([
    ("T006", "C102", 2100, "FAILED"),
    ("T007", "C104", 4500, "FAILED"),
    ("T008", "C106", 5100, "SUCCESS"),
    ("T009", "C103", 1400, "FAILED"),
    ("T010", "C107", 3000, "FAILED")
])

# REQUIREMENT:
#
# 1. Combine both days of transaction data.
# 2. Keep only FAILED transactions.
# 3. Extract customer IDs.
# 4. A customer may fail multiple times.
# 5. Return every failed customer only once.
#
# Expected output:
#
# C102
# C103
# C104
# C107


# ============================================================
# QUESTION 2
# Successful Customers Present in Both Months
# Level: MEDIUM
# ============================================================

july = sc.parallelize([
    ("T001", "C101", 1200, "SUCCESS"),
    ("T002", "C102", 3000, "FAILED"),
    ("T003", "C103", 2500, "SUCCESS"),
    ("T004", "C104", 1800, "SUCCESS"),
    ("T005", "C101", 900,  "SUCCESS"),
    ("T006", "C105", 5000, "SUCCESS")
])

august = sc.parallelize([
    ("T101", "C101", 2000, "SUCCESS"),
    ("T102", "C103", 1500, "FAILED"),
    ("T103", "C104", 3500, "SUCCESS"),
    ("T104", "C106", 4200, "SUCCESS"),
    ("T105", "C105", 2200, "SUCCESS"),
    ("T106", "C107", 9000, "SUCCESS")
])

# REQUIREMENT:
#
# Find customers who had at least one SUCCESSFUL transaction
# in BOTH July and August.
#
# A customer may have multiple transactions in a month.
#
# Expected:
#
# C101
# C104
# C105


# ============================================================
# QUESTION 3
# Customers Who Stopped Transacting
# Level: MEDIUM
# ============================================================

july_transactions = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T003", "C103", 3000),
    ("T004", "C101", 1500),
    ("T005", "C104", 4000),
    ("T006", "C105", 5000)
])

august_transactions = sc.parallelize([
    ("T101", "C101", 2500),
    ("T102", "C103", 3500),
    ("T103", "C106", 6000),
    ("T104", "C103", 1200),
    ("T105", "C105", 1700)
])

# REQUIREMENT:
#
# Find customers who were active in July
# but had NO transaction at all in August.
#
# Expected:
#
# C102
# C104


# ============================================================
# QUESTION 4
# High-Value International Spend Per Customer
# Level: MEDIUM-HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "INDIA",   "SUCCESS"),
    ("T002", "C101", 45000, "USA",     "SUCCESS"),
    ("T003", "C102", 28000, "UK",      "SUCCESS"),
    ("T004", "C102", 32000, "USA",     "FAILED"),
    ("T005", "C101", 18000, "UAE",     "SUCCESS"),
    ("T006", "C103", 70000, "INDIA",   "SUCCESS"),
    ("T007", "C103", 25000, "GERMANY", "SUCCESS"),
    ("T008", "C104", 9000,  "USA",     "SUCCESS"),
    ("T009", "C104", 27000, "UK",      "SUCCESS"),
    ("T010", "C105", 56000, "UAE",     "FAILED")
])

# REQUIREMENT:
#
# Calculate total international spend for every customer.
#
# Conditions:
#
# status must be SUCCESS
# country must NOT be INDIA
# amount must be >= 10000
#
# Expected:
#
# C101 -> 63000
# C102 -> 28000
# C103 -> 25000
# C104 -> 27000


# ============================================================
# QUESTION 5
# Customers Crossing Spending Threshold
# Level: MEDIUM-HARD
# ============================================================

transactions = sc.parallelize([
    ("C101", 12000),
    ("C102", 8000),
    ("C101", 18000),
    ("C103", 25000),
    ("C102", 7000),
    ("C101", 22000),
    ("C103", 9000),
    ("C104", 45000),
    ("C104", 10000),
    ("C105", 15000)
])

# REQUIREMENT:
#
# Calculate total spend per customer.
#
# Then return only customers whose TOTAL spend
# is >= 40000.
#
# Expected:
#
# C101 -> 52000
# C104 -> 55000


# ============================================================
# QUESTION 6
# Source vs Target Reconciliation
# Level: HARD
# ============================================================

source = sc.parallelize([
    ("T001", "C101", 1000, "SUCCESS"),
    ("T002", "C102", 2000, "SUCCESS"),
    ("T003", "C103", 3000, "FAILED"),
    ("T004", "C104", 4000, "SUCCESS"),
    ("T005", "C105", 5000, "SUCCESS"),
    ("T006", "C106", 6000, "SUCCESS"),
    ("T007", "C107", 7000, "FAILED")
])

target = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T004", "C104", 4000),
    ("T008", "C108", 8000)
])

# REQUIREMENT:
#
# Target should contain ONLY successful source transactions.
#
# Find three outputs:
#
# 1. MATCHED transaction IDs
#    Successful source IDs that exist in target.
#
# 2. MISSING transaction IDs
#    Successful source IDs that do NOT exist in target.
#
# 3. EXTRA transaction IDs
#    IDs present in target but not present in successful source.
#
# Expected:
#
# MATCHED
# T001
# T002
# T004
#
# MISSING
# T005
# T006
#
# EXTRA
# T008


# ============================================================
# QUESTION 7
# Unique Error Applications Across Two Servers
# Level: HARD
# ============================================================

server1_logs = sc.parallelize([
    ("2026-08-18 10:01", "PAYMENT", "ERROR",   "Timeout"),
    ("2026-08-18 10:02", "AUTH",    "INFO",    "Login"),
    ("2026-08-18 10:03", "ORDER",   "ERROR",   "Database unavailable"),
    ("2026-08-18 10:04", "PAYMENT", "ERROR",   "HTTP 500"),
    ("2026-08-18 10:05", "SEARCH",  "WARNING", "Slow response")
])

server2_logs = sc.parallelize([
    ("2026-08-18 10:06", "AUTH",    "ERROR", "Invalid token"),
    ("2026-08-18 10:07", "PAYMENT", "ERROR", "Connection refused"),
    ("2026-08-18 10:08", "ORDER",   "INFO",  "Order created"),
    ("2026-08-18 10:09", "PROFILE", "ERROR", "Service unavailable")
])

# REQUIREMENT:
#
# Combine logs from both servers.
#
# Keep ERROR logs only.
#
# Return application names that generated errors.
#
# Application name should appear only once.
#
# Expected:
#
# PAYMENT
# ORDER
# AUTH
# PROFILE


# ============================================================
# QUESTION 8
# Count Errors Per Application Across Multiple Servers
# Level: HARD
# ============================================================

server1_logs = sc.parallelize([
    ("PAYMENT", "ERROR"),
    ("AUTH",    "SUCCESS"),
    ("ORDER",   "ERROR"),
    ("PAYMENT", "ERROR"),
    ("SEARCH",  "SUCCESS")
])

server2_logs = sc.parallelize([
    ("AUTH",    "ERROR"),
    ("PAYMENT", "ERROR"),
    ("ORDER",   "SUCCESS"),
    ("PROFILE", "ERROR"),
    ("PAYMENT", "SUCCESS")
])

server3_logs = sc.parallelize([
    ("PAYMENT", "ERROR"),
    ("AUTH",    "ERROR"),
    ("ORDER",   "ERROR"),
    ("PROFILE", "SUCCESS")
])

# REQUIREMENT:
#
# Combine logs from all three servers.
#
# Keep ERROR records only.
#
# Calculate number of errors per application.
#
# Expected:
#
# PAYMENT -> 4
# AUTH    -> 2
# ORDER   -> 2
# PROFILE -> 1


# ============================================================
# QUESTION 9
# Group Transactions Into Risk Categories
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 5000,   "INDIA"),
    ("T002", "C102", 25000,  "USA"),
    ("T003", "C103", 75000,  "UK"),
    ("T004", "C104", 110000, "UAE"),
    ("T005", "C105", 45000,  "INDIA"),
    ("T006", "C106", 95000,  "GERMANY"),
    ("T007", "C107", 8000,   "USA"),
    ("T008", "C108", 55000,  "INDIA"),
    ("T009", "C109", 150000, "USA")
])

# REQUIREMENT:
#
# Ignore transactions from INDIA.
#
# For remaining transactions, group the COMPLETE records
# into risk categories based on amount.
#
# amount < 10000
#     LOW
#
# amount >= 10000 and amount < 50000
#     MEDIUM
#
# amount >= 50000 and amount < 100000
#     HIGH
#
# amount >= 100000
#     CRITICAL
#
# Expected conceptual output:
#
# LOW
#     T007
#
# MEDIUM
#     T002
#
# HIGH
#     T003
#     T006
#
# CRITICAL
#     T004
#     T009


# ============================================================
# QUESTION 10
# Customer Transaction Summary
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 10000, "SUCCESS"),
    ("T002", "C102", 8000,  "SUCCESS"),
    ("T003", "C101", 15000, "FAILED"),
    ("T004", "C103", 25000, "SUCCESS"),
    ("T005", "C101", 22000, "SUCCESS"),
    ("T006", "C102", 7000,  "SUCCESS"),
    ("T007", "C103", 9000,  "FAILED"),
    ("T008", "C104", 45000, "SUCCESS"),
    ("T009", "C101", 13000, "SUCCESS"),
    ("T010", "C104", 5000,  "SUCCESS")
])

# REQUIREMENT:
#
# Consider only SUCCESS transactions.
#
# For every customer calculate:
#
# 1. Total successful transaction amount
# 2. Number of successful transactions
#
# Expected:
#
# C101 -> (45000, 3)
# C102 -> (15000, 2)
# C103 -> (25000, 1)
# C104 -> (50000, 2)


# ============================================================
# QUESTION 11
# Active Fraud-Watch Customers With High Spend
# Level: HARD
# ============================================================

fraud_watchlist = sc.parallelize([
    "C101",
    "C103",
    "C105",
    "C108",
    "C110"
])

transactions = sc.parallelize([
    ("T001", "C101", 15000, "SUCCESS"),
    ("T002", "C102", 50000, "SUCCESS"),
    ("T003", "C101", 30000, "SUCCESS"),
    ("T004", "C103", 12000, "FAILED"),
    ("T005", "C103", 45000, "SUCCESS"),
    ("T006", "C104", 70000, "SUCCESS"),
    ("T007", "C105", 20000, "SUCCESS"),
    ("T008", "C105", 25000, "SUCCESS"),
    ("T009", "C108", 10000, "FAILED"),
    ("T010", "C108", 18000, "SUCCESS")
])

# REQUIREMENT:
#
# 1. Consider only successful transactions.
# 2. Calculate total spend per customer.
# 3. Keep customers whose total spend >= 40000.
# 4. From those customers, return only customers who are
#    present in the fraud watchlist.
#
# Expected:
#
# C101
# C103
# C105


# ============================================================
# QUESTION 12
# Find Customers New This Month
# Level: HARD
# ============================================================

july = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 3000),
    ("T003", "C103", 2500),
    ("T004", "C101", 1800),
    ("T005", "C104", 5000)
])

august = sc.parallelize([
    ("T101", "C101", 2000),
    ("T102", "C103", 4000),
    ("T103", "C105", 3500),
    ("T104", "C106", 1000),
    ("T105", "C105", 2500),
    ("T106", "C107", 9000)
])

# REQUIREMENT:
#
# Find customers who appear in August but never appeared in July.
#
# A customer may have multiple transactions.
#
# Expected:
#
# C105
# C106
# C107


# ============================================================
# QUESTION 13
# Customers Active in Either Month But Not Both
# Level: HARD
# ============================================================

july = sc.parallelize([
    ("C101", 1000),
    ("C102", 2000),
    ("C103", 3000),
    ("C104", 4000),
    ("C101", 5000)
])

august = sc.parallelize([
    ("C101", 2000),
    ("C103", 3500),
    ("C105", 6000),
    ("C106", 8000),
    ("C105", 1200)
])

# REQUIREMENT:
#
# Find customers who were active in ONLY ONE of the two months.
#
# In other words:
#
# active in July but not August
#
# OR
#
# active in August but not July
#
# Expected:
#
# C102
# C104
# C105
# C106


# ============================================================
# QUESTION 14
# Failed Transaction Amount Per Customer
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "SUCCESS"),
    ("T002", "C101", 5000,  "FAILED"),
    ("T003", "C102", 9000,  "FAILED"),
    ("T004", "C101", 7000,  "FAILED"),
    ("T005", "C103", 15000, "SUCCESS"),
    ("T006", "C102", 4000,  "FAILED"),
    ("T007", "C104", 20000, "FAILED"),
    ("T008", "C103", 8000,  "FAILED"),
    ("T009", "C104", 10000, "SUCCESS")
])

# REQUIREMENT:
#
# Consider only FAILED transactions.
#
# Calculate total failed transaction amount per customer.
#
# Then return only customers whose total FAILED amount
# is >= 10000.
#
# Expected:
#
# C101 -> 12000
# C102 -> 13000
# C104 -> 20000


# ============================================================
# QUESTION 15
# Detect Duplicate Transaction IDs
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T003", "C103", 3000),
    ("T001", "C101", 1000),
    ("T004", "C104", 4000),
    ("T002", "C102", 2000),
    ("T005", "C105", 5000),
    ("T001", "C101", 1000)
])

# REQUIREMENT:
#
# Find transaction IDs that occur MORE THAN ONCE.
#
# Output should also contain occurrence count.
#
# Expected:
#
# T001 -> 3
# T002 -> 2
#
# Do not solve this only by using Python collections.
# Use RDD transformations.


# ============================================================
# QUESTION 16
# Group S3 Files By Year-Month After Filtering
# Level: HARD
# ============================================================

paths = sc.parallelize([
    "s3://company/sales/year=2026/month=08/day=01/file1.parquet",
    "s3://company/sales/year=2026/month=08/day=02/file2.parquet",
    "s3://company/sales/year=2026/month=07/day=31/file3.parquet",
    "s3://company/logs/year=2026/month=08/day=01/log1.json",
    "s3://company/sales/year=2025/month=12/day=01/file4.parquet",
    "s3://company/logs/year=2025/month=12/day=02/log2.json",
    "s3://company/sales/year=2026/month=08/day=03/file5.parquet"
])

# REQUIREMENT:
#
# 1. Keep only paths belonging to /sales/.
# 2. Extract year and month.
# 3. Extract file name.
# 4. Group files by YYYY-MM.
#
# Expected conceptual output:
#
# 2026-08
#     file1.parquet
#     file2.parquet
#     file5.parquet
#
# 2026-07
#     file3.parquet
#
# 2025-12
#     file4.parquet


# ============================================================
# QUESTION 17
# Find Customers With Both Success and Failure
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", "SUCCESS"),
    ("T002", "C101", "FAILED"),
    ("T003", "C102", "SUCCESS"),
    ("T004", "C103", "FAILED"),
    ("T005", "C103", "FAILED"),
    ("T006", "C104", "SUCCESS"),
    ("T007", "C104", "FAILED"),
    ("T008", "C105", "SUCCESS"),
    ("T009", "C105", "SUCCESS"),
    ("T010", "C106", "FAILED")
])

# REQUIREMENT:
#
# Find customers who have at least:
#
# one SUCCESS transaction
#
# AND
#
# one FAILED transaction.
#
# Expected:
#
# C101
# C104


# ============================================================
# QUESTION 18
# Transaction Volume By Country
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "INDIA",   "SUCCESS"),
    ("T002", "C102", 22000, "USA",     "SUCCESS"),
    ("T003", "C103", 5000,  "INDIA",   "FAILED"),
    ("T004", "C104", 32000, "USA",     "SUCCESS"),
    ("T005", "C105", 45000, "UK",      "SUCCESS"),
    ("T006", "C106", 8000,  "UK",      "SUCCESS"),
    ("T007", "C107", 51000, "USA",     "FAILED"),
    ("T008", "C108", 27000, "GERMANY", "SUCCESS"),
    ("T009", "C109", 18000, "INDIA",   "SUCCESS"),
    ("T010", "C110", 15000, "UK",      "SUCCESS")
])

# REQUIREMENT:
#
# Consider only SUCCESS transactions.
#
# For each country calculate:
#
# 1. Total successful transaction amount
# 2. Number of successful transactions
#
# Expected:
#
# INDIA   -> (30000, 2)
# USA     -> (54000, 2)
# UK      -> (68000, 3)
# GERMANY -> (27000, 1)


# ============================================================
# QUESTION 19
# High-Risk Customers Seen Across Two Systems
# Level: VERY HARD
# ============================================================

banking_system = sc.parallelize([
    ("C101", 120000),
    ("C102", 45000),
    ("C103", 95000),
    ("C104", 150000),
    ("C105", 70000),
    ("C106", 125000)
])

credit_card_system = sc.parallelize([
    ("C101", 40000),
    ("C103", 30000),
    ("C104", 60000),
    ("C105", 50000),
    ("C107", 140000),
    ("C108", 160000)
])

# REQUIREMENT:
#
# A customer is HIGH RISK if their total amount
# across BOTH systems is >= 150000.
#
# Customers can appear in one system or both.
#
# Calculate combined exposure per customer.
#
# Then return only HIGH-RISK customers.
#
# Expected:
#
# C101 -> 160000
# C104 -> 210000


# ============================================================
# QUESTION 20
# Complete Daily Reconciliation
# Level: VERY HARD / INTERVIEW LEVEL
# ============================================================

source_day1 = sc.parallelize([
    ("T001", "C101", 10000, "SUCCESS"),
    ("T002", "C102", 15000, "SUCCESS"),
    ("T003", "C103", 8000,  "FAILED"),
    ("T004", "C104", 22000, "SUCCESS")
])

source_day2 = sc.parallelize([
    ("T005", "C101", 12000, "SUCCESS"),
    ("T006", "C105", 30000, "SUCCESS"),
    ("T007", "C106", 9000,  "FAILED"),
    ("T008", "C102", 18000, "SUCCESS"),
    ("T009", "C107", 45000, "SUCCESS")
])

target = sc.parallelize([
    ("T001", "C101", 10000),
    ("T002", "C102", 15000),
    ("T004", "C104", 22000),
    ("T005", "C101", 12000),
    ("T008", "C102", 18000),
    ("T010", "C108", 50000)
])

# REQUIREMENT:
#
# PART 1:
# Combine source Day 1 and Day 2.
#
# PART 2:
# Keep only SUCCESSFUL source transactions.
#
# PART 3:
# Find:
#
#     MATCHED transaction IDs
#     MISSING transaction IDs
#     EXTRA transaction IDs
#
# PART 4:
# For the SUCCESSFUL source data,
# calculate total successful amount per customer.
#
# PART 5:
# Return customers whose successful source total
# is >= 30000.
#
#
# Expected reconciliation:
#
# MATCHED
# T001
# T002
# T004
# T005
# T008
#
# MISSING
# T006
# T009
#
# EXTRA
# T010
#
#
# Expected high-value customers:
#
# C102 -> 33000
# C105 -> 30000
# C107 -> 45000